# HIPE-2026 Spatial Relation Extraction (LLaMA 3B, No Reasoning)

This notebook presents an end-to-end pipeline for predicting spatial and temporal relations (`at` and `isAt`) between Person and Location entities. It leverages a large language model (LLaMA 3.2 3B) dynamically paired with task-specific LoRA adapters to perform specialized sequential inference without reasoning steps. The pipeline encompasses environment setup, robust output parsing, inference execution, and a comprehensive evaluation against a gold-standard dataset across multiple languages.

### 1. Environment & Setup
Unsloth 'Nuclear' installation and Google Drive mounting.

In [ ]:
# Unsloth "Nuclear" installation script
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9zsir7x6/unsloth_5ff2855eebbf428fbfe08b9719494c4c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9zsir7x6/unsloth_5ff2855eebbf428fbfe08b9719494c4c
  Resolved https://github.com/unslothai/unsloth.git to commit c8bcacc3fea27e97bea0ea09c9ad7554c729724f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.6 MB/s eta 0:00:00
  

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import re

# Base Paths
BASE_PATH = "/content/drive/MyDrive/colab_data/HIPE-2026-data"
AT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_3b_at_adapter_v2")
ISAT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_3b_isAt_adapter_v2")
DATA_PATH = os.path.join(BASE_PATH, "data/sandbox")

LANGUAGES = ['en', 'de', 'fr']

### 2. Specialized Logic Requirements
Robust harvester and data loading utilities.

In [ ]:
def harvest_json_robust(text):
    # Normalize smart quotes
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

    # Extract JSON objects
    matches = re.findall(r'\{[^{}]*\}', text)
    results = []
    for match in matches:
        # Correct unquoted labels
        match = re.sub(r':\s*TRUE\b', ': "TRUE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*FALSE\b', ': "FALSE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*PROBABLE\b', ': "PROBABLE"', match, flags=re.IGNORECASE)
        try:
            results.append(json.loads(match))
        except json.JSONDecodeError:
            pass

    return results if results else [{"label": "ERROR"}]

def load_data(lang):
    filepath = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")
    data = []
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    else:
        print(f"Warning: {filepath} not found.")
    return data

### 3. Prompt Templates
Defining the prompt templates matching the training setup.

In [ ]:
at_definitions = """• at=TRUE: Explicit evidence of residency, origin, or long-term role.
• at=PROBABLE: Implicit cues or professional affiliation make a relation a likely assumption.
• at=FALSE: No evidence or contradictory evidence."""

isAt_definitions = """• isAt=TRUE: Explicit evidence the person was at the location within one month of the publication date.
• isAt=FALSE: Event occurred in the past or person is elsewhere."""

base_prompt = """TASK: Classify {relation} between Person: {person} and Place: {place}.
DEFINITIONS: ### DEFINITIONS:
{definitions}
TEXT: {text}"""

def format_chat_prompt(relation, person, place, text):
    definitions = at_definitions if relation == 'at' else isAt_definitions
    user_msg = base_prompt.format(
        relation=relation,
        person=person,
        place=place,
        definitions=definitions,
        text=text
    )
    return [{"role": "user", "content": user_msg}]

### 4. Sequential Inference
Loading base model, `at` inference, followed by adapter switch to `isAt` and Logic Guard.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys

max_seq_length = 4096

if not os.path.exists(AT_ADAPTER):
    print(f"Error: AT adapter path does not exist: {AT_ADAPTER}")
    sys.exit(1)

# Load base model
print("Loading Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 1. AT Inference
print(f"Loading AT-Specialist from {AT_ADAPTER}...")
model.load_adapter(AT_ADAPTER, adapter_name="at_adapter")
model.set_adapter("at_adapter")
FastLanguageModel.for_inference(model)

lang = "fr"

if 'at_predictions' not in globals():
    at_predictions = {l: [] for l in LANGUAGES}
else:
    at_predictions[lang] = [] # Reset predictions for the selected language if re-running

print(f"Running AT inference for {lang}...")
data = load_data(lang)
for item in data:
    # Iterate through the pairs inside the document!
    for pair in item.get('sampled_pairs', []):
        # FIX: Extract from correct list keys
        pers_list = pair.get('pers_mentions_list', [])
        loc_list = pair.get('loc_mentions_list', [])
        person = pers_list[0] if pers_list else ""
        place = loc_list[0] if loc_list else ""

        messages = format_chat_prompt('at', person, place, item.get('text', ''))
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

        outputs = model.generate(input_ids=inputs, max_new_tokens=64, do_sample=False)
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

        # FIX: Check for raw string output first, fallback to JSON parsing
        resp_clean = response.strip().upper()
        if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
            pred = resp_clean
        else:
            parsed = harvest_json_robust(response)
            pred = parsed[0].get('label', 'ERROR') if parsed else 'ERROR'

        # Save prediction directly to the standard key
        pair['at'] = pred

    at_predictions[lang].append(item)

Loading Base Model...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Loading AT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_at_adapter_v2...


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running AT inference for fr...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6

In [ ]:
import torch
import gc

# Delete the model and trainer from memory
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Now load the base model fresh
print("Loading Base Model for ISAT...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Load the isAt adapter
print(f"Loading ISAT-Specialist from {ISAT_ADAPTER}...")
model.load_adapter(ISAT_ADAPTER, adapter_name="isAt_adapter")
model.set_adapter("isAt_adapter")
FastLanguageModel.for_inference(model)

print("Ready to run ISAT inference loop.")

Loading Base Model for ISAT...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Loading ISAT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter_v2...


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Ready to run ISAT inference loop.


In [ ]:
# 2. ISAT Inference
lang = "fr"

if 'final_results' not in globals():
    final_results = {l: [] for l in LANGUAGES}
else:
    final_results[lang] = []

print(f"Running ISAT inference for {lang}...")
for item in at_predictions[lang]:
    # Iterate through the pairs inside the document!
    for pair in item.get('sampled_pairs', []):
        # FIX: Extract from correct list keys
        pers_list = pair.get('pers_mentions_list', [])
        loc_list = pair.get('loc_mentions_list', [])
        person = pers_list[0] if pers_list else ""
        place = loc_list[0] if loc_list else ""

        messages = format_chat_prompt('isAt', person, place, item.get('text', ''))
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

        outputs = model.generate(input_ids=inputs, max_new_tokens=64, do_sample=False)
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

        # FIX: Check for raw string output first, fallback to JSON parsing
        resp_clean = response.strip().upper()
        if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
            isAt_pred = resp_clean
        else:
            parsed = harvest_json_robust(response)
            isAt_pred = parsed[0].get('label', 'ERROR') if parsed else 'ERROR'

        # Save prediction directly to the standard key
        pair['isAt'] = isAt_pred

    final_results[lang].append(item)

# Save Integrated Output
out_path = os.path.join(BASE_PATH, f"integrated_llama_3b_{lang}_v2_results.jsonl")
with open(out_path, 'w', encoding='utf-8') as f:
    for res in final_results[lang]:
        f.write(json.dumps(res) + '\n')
print(f"Saved final predictions to {out_path}")

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running ISAT inference for fr...


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved final predictions to /content/drive/MyDrive/colab_data/HIPE-2026-data/integrated_llama_3b_fr_v2_results.jsonl


### 5. Evaluation
Automatically trigger the official scorer scripts for each integrated output file.

In [ ]:
import json
import os

print("Running automated evaluation script...")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_3b_{lang}_v2_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if os.path.exists(pred_path) and os.path.exists(gold_path):
        print(f"\nEvaluating {lang}...")
        # cd into BASE_PATH so the script can find the 'schemas/' directory
        !cd "{BASE_PATH}" && python scripts/file_scorer_evaluation.py --predictions_file "{pred_path}" --gold_data_file "{gold_path}"
    else:
        print(f"Skipping eval for {lang}. Check if prediction or gold files exist.")

Running automated evaluation script...

Evaluating en...

Evaluation Results for integrated_llama_3b_en_v2_results.jsonl:
  'at': macro_recall=0.3158, accuracy=0.2914 (44/151)
  'isAt': macro_recall=0.5382, accuracy=0.8212 (124/151)
  'global': macro_recall=0.4270 (168/302)


Evaluating de...

Evaluation Results for integrated_llama_3b_de_v2_results.jsonl:
  'at': macro_recall=0.2666, accuracy=0.3356 (145/432)
  'isAt': macro_recall=0.4944, accuracy=0.7731 (334/432)
  'global': macro_recall=0.3805 (479/864)


Evaluating fr...

Evaluation Results for integrated_llama_3b_fr_v2_results.jsonl:
  'at': macro_recall=0.3498, accuracy=0.4272 (640/1498)
  'isAt': macro_recall=0.5630, accuracy=0.7690 (1152/1498)
  'global': macro_recall=0.4564 (1792/2996)



In [ ]:
import json
import os

print("Detailed Prediction Analysis by Label")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_3b_{lang}_v2_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not (os.path.exists(pred_path) and os.path.exists(gold_path)):
        print(f"Missing files for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Analysis for {lang.upper()} ---")
    print(f"{'='*40}")

    # 1. Load gold data into a dictionary mapped by document_id
    gold_data = {}
    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            # Map pair entities to their gold pairs so order doesn't matter
            gold_data[doc_id] = {}
            for pair in item.get('sampled_pairs', []):
                pers_id = pair.get('pers_entity_id')
                loc_id = pair.get('loc_entity_id')
                gold_data[doc_id][(pers_id, loc_id)] = pair

    # 2. Track stats
    at_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0},
        "PROBABLE": {"correct": 0, "wrong": 0}
    }
    isat_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0}
    }

    # 3. Compare predictions against gold
    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            pred_pairs = item.get('sampled_pairs', [])

            for p_pair in pred_pairs:
                pers_id = p_pair.get('pers_entity_id')
                loc_id = p_pair.get('loc_entity_id')

                # Find corresponding gold pair
                g_pair = gold_data.get(doc_id, {}).get((pers_id, loc_id))

                if not g_pair:
                    continue # Skip if no matching gold pair found

                # --- Check 'at' Field ---
                g_at = g_pair.get('at', 'FALSE')
                p_at = p_pair.get('at', 'ERROR')

                if g_at in at_stats:
                    if g_at == p_at:
                        at_stats[g_at]['correct'] += 1
                    else:
                        at_stats[g_at]['wrong'] += 1

                # --- Check 'isAt' Field ---
                g_isat = g_pair.get('isAt', 'FALSE')
                p_isat = p_pair.get('isAt', 'ERROR')

                if g_isat in isat_stats:
                    if g_isat == p_isat:
                        isat_stats[g_isat]['correct'] += 1
                    else:
                        isat_stats[g_isat]['wrong'] += 1

    # 4. Print Results
    print("\n[ 'AT' FIELD STATS ]")
    for label, counts in at_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Accuracy: {acc:>5.1f}%")

    print("\n[ 'ISAT' FIELD STATS ]")
    for label, counts in isat_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Accuracy: {acc:>5.1f}%")


Detailed Prediction Analysis by Label

--- Analysis for EN ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   15 Correct |   14 Wrong | Accuracy:  51.7%
  Gold=FALSE   :   28 Correct |   40 Wrong | Accuracy:  41.2%
  Gold=PROBABLE:    1 Correct |   53 Wrong | Accuracy:   1.9%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :    3 Correct |   15 Wrong | Accuracy:  16.7%
  Gold=FALSE   :  121 Correct |   12 Wrong | Accuracy:  91.0%

--- Analysis for DE ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :    2 Correct |   39 Wrong | Accuracy:   4.9%
  Gold=FALSE   :   82 Correct |  162 Wrong | Accuracy:  33.6%
  Gold=PROBABLE:   61 Correct |   86 Wrong | Accuracy:  41.5%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :    5 Correct |   24 Wrong | Accuracy:  17.2%
  Gold=FALSE   :  329 Correct |   74 Wrong | Accuracy:  81.6%

--- Analysis for FR ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   54 Correct |  125 Wrong | Accuracy:  30.2%
  Gold=FALSE   :  507 Correct |  445 Wrong | Accuracy:  53.3%
  Gold=PROBABLE:   79 Correct

In [ ]:
import json
import os

print("Prediction Distribution (Model Guesses)")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_3b_{lang}_v2_results.jsonl")

    if not os.path.exists(pred_path):
        print(f"Missing prediction file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Model Guesses for {lang.upper()} ---")
    print(f"{'='*40}")

    at_guesses = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_guesses = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                p_at = pair.get('at', 'ERROR')
                p_isat = pair.get('isAt', 'ERROR')

                if p_at in at_guesses:
                    at_guesses[p_at] += 1
                else:
                    at_guesses['ERROR'] += 1

                if p_isat in isat_guesses:
                    isat_guesses[p_isat] += 1
                else:
                    isat_guesses['ERROR'] += 1

    print("\n[ 'AT' FIELD GUESSES ]")
    for label, count in at_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GUESSES ]")
    for label, count in isat_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

Prediction Distribution (Model Guesses)

--- Model Guesses for EN ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :   87 times
  Guessed FALSE   :   59 times
  Guessed PROBABLE:    5 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   15 times
  Guessed FALSE   :  136 times

--- Model Guesses for DE ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :   76 times
  Guessed FALSE   :  169 times
  Guessed PROBABLE:  187 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   79 times
  Guessed FALSE   :  353 times

--- Model Guesses for FR ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :  426 times
  Guessed FALSE   :  782 times
  Guessed PROBABLE:  290 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :  299 times
  Guessed FALSE   : 1199 times


In [ ]:
import json
import os

print("Gold Label Distribution (Actual Data)")

for lang in LANGUAGES:
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not os.path.exists(gold_path):
        print(f"Missing gold file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Gold Labels for {lang.upper()} ---")
    print(f"{'='*40}")

    at_counts = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_counts = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                g_at = pair.get('at', 'ERROR')
                g_isat = pair.get('isAt', 'ERROR')

                if g_at in at_counts:
                    at_counts[g_at] += 1
                else:
                    at_counts['ERROR'] += 1

                if g_isat in isat_counts:
                    isat_counts[g_isat] += 1
                else:
                    isat_counts['ERROR'] += 1

    print("\n[ 'AT' FIELD GOLD LABELS ]")
    for label, count in at_counts.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GOLD LABELS ]")
    for label, count in isat_counts.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

Gold Label Distribution (Actual Data)

--- Gold Labels for EN ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :   68 times
  Actual PROBABLE:   54 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   18 times
  Actual FALSE   :  133 times

--- Gold Labels for DE ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   41 times
  Actual FALSE   :  244 times
  Actual PROBABLE:  147 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :  403 times

--- Gold Labels for FR ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :  179 times
  Actual FALSE   :  952 times
  Actual PROBABLE:  367 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :  127 times
  Actual FALSE   : 1371 times


### 6. Conclusion
The evaluation sections above break down the model's predictive performance against the gold labels. By splitting the dual inference tasks (`at` and `isAt`) into separate LoRA adapters dynamically loaded into memory, this approach efficiently handles multiple classification constraints without exceeding GPU memory limits. The detailed breakdown provides insights into the true/false/probable label distributions and highlights areas where the adapter excels or struggles.